# 📊 Samsung SmartThings AI Energy Assistant - Exploratory Data Analysis (EDA)

This notebook executes a comprehensive Exploratory Data Analysis on the 30-day IoT time-series dataset (`database/energy_usage.csv`) generated for **HOUSE001** (a 2-Bedroom Apartment in Chennai, Tamil Nadu, India).

### Objectives:
- Inspect schema, data integrity, and dataset completeness.
- Quantify time-series consumption patterns, room distribution, and appliance rankings.
- Evaluate environmental impacts (climate, temperature, weather) and occupancy correlations.
- Analyze daily energy threshold violations and financial cost metrics.
- Generate **Dashboard Insights** for frontend UI display and **AI Insights** for LLM recommendation systems.



## 0. Environment Setup & Data Ingestion


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Visual styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 100
sns.set_palette("muted")

# Load dataset
CSV_PATH = "../energy_usage.csv"
df_raw = pd.read_csv(CSV_PATH)
df = df_raw.copy()

print("Dataset successfully loaded from:", CSV_PATH)



## 1. Dataset Overview

In this section, we examine the structural dimensions, row count, column count, memory footprint, and preview head/tail rows of the raw IoT dataset.



In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
print("\n--- First 10 Rows ---")
display(df.head(10))

print("\n--- Last 10 Rows ---")
display(df.tail(10))



## 2. Data Types & Type Conversion

We identify the inferred data types for each column and perform non-destructive type conversions (e.g. converting `timestamp` to `datetime` objects and boolean/categorical flags) for analytical precision without altering the original CSV.



In [ ]:
print("=== INITIAL DATA TYPES ===")
print(df.dtypes)

# Perform type conversions for in-memory analysis only
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['date'] = pd.to_datetime(df['date'])
df['is_weekend'] = df['is_weekend'].astype(str).str.upper() == 'TRUE'
df['occupancy'] = df['occupancy'].astype(str).str.upper() == 'TRUE'
df['threshold_exceeded'] = df['threshold_exceeded'].astype(str).str.upper() == 'TRUE'

# Clean numeric conversions if needed
numeric_cols = ['hour', 'rated_power_watts', 'runtime_minutes', 'power_consumption_wh', 
                'energy_kwh', 'electricity_cost', 'ambient_temperature', 'daily_limit_kwh']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n=== CONVERTED DATA TYPES FOR ANALYSIS ===")
print(df.dtypes)



## 3. Missing Value Analysis

Checking for missing (NaN/Null) values across all features to guarantee complete IoT time-series coverage.



In [ ]:
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
print("=== MISSING VALUE SUMMARY ===")
display(missing_df)

# Visualize missing values heatmap
plt.figure(figsize=(10, 3))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing Value Scan (Yellow indicates missing values)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()



## 4. Duplicate Analysis

Verifying data integrity by checking for duplicate rows, timestamp collisions per appliance, and unique appliance count.



In [ ]:
duplicate_rows = df.duplicated().sum()
ts_appliance_dups = df.duplicated(subset=['timestamp', 'appliance_id']).sum()
unique_appliances = df['appliance_id'].nunique()
unique_rooms = df['room_id'].nunique()

print("=== DUPLICATE & INTEGRITY SCAN ===")
print(f"Duplicate Rows Count: {duplicate_rows}")
print(f"Duplicate Appliance-Timestamp Readings: {ts_appliance_dups}")
print(f"Unique Appliances Monitored: {unique_appliances}")
print(f"Unique Rooms Monitored: {unique_rooms}")



## 5. Time-Series Analysis

Analyzing hourly energy consumption trends, daily consumption totals, weekly patterns, and peak load hours across the 30-day period.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 5.1 Hourly Average Energy Consumption
hourly_energy = df.groupby('hour')['energy_kwh'].sum() / 30.0 # Average per day
sns.lineplot(x=hourly_energy.index, y=hourly_energy.values, ax=axes[0, 0], marker='o', color='#0a60ff', linewidth=2.5)
axes[0, 0].set_title('Average Hourly Household Energy Consumption (kWh)', fontweight='bold')
axes[0, 0].set_xlabel('Hour of Day (0-23)')
axes[0, 0].set_ylabel('Energy (kWh)')
axes[0, 0].set_xticks(range(0, 24, 2))

# 5.2 Daily Household Energy Consumption
daily_energy = df.groupby('date')['energy_kwh'].sum()
sns.barplot(x=daily_energy.index.strftime('%d-%b'), y=daily_energy.values, ax=axes[0, 1], color='#4f46e5')
axes[0, 1].set_title('Total Daily Household Consumption (30 Days)', fontweight='bold')
axes[0, 1].set_xlabel('Date')
axes[0, 1].set_ylabel('Total Energy (kWh)')
axes[0, 1].tick_params(axis='x', rotation=90)

# 5.3 Weekly Consumption Trend
df['week'] = df['date'].dt.isocalendar().week
weekly_trend = df.groupby('day_of_week')['energy_kwh'].sum()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_trend = weekly_trend.reindex(days_order)
sns.barplot(x=weekly_trend.index, y=weekly_trend.values, ax=axes[1, 0], palette='Blues_r')
axes[1, 0].set_title('Consumption by Day of Week', fontweight='bold')
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Cumulative Energy (kWh)')

# 5.4 Peak Usage Hours Distribution
peak_hours = df.groupby('hour')['power_consumption_wh'].mean()
sns.barplot(x=peak_hours.index, y=peak_hours.values, ax=axes[1, 1], color='#e11d48')
axes[1, 1].set_title('Household Average Power Load Profile (Wh per Hour)', fontweight='bold')
axes[1, 1].set_xlabel('Hour of Day')
axes[1, 1].set_ylabel('Mean Wh Draw')

plt.tight_layout()
plt.show()



## 6. Room-wise Analysis

Evaluating total energy consumption, average hourly load draw, and cumulative electricity billing costs broken down by each room.



In [ ]:
room_stats = df.groupby('room_name').agg(
    Total_Energy_kWh=('energy_kwh', 'sum'),
    Avg_Hourly_Wh=('power_consumption_wh', 'mean'),
    Total_Cost_INR=('electricity_cost', 'sum')
).reset_index().sort_values(by='Total_Energy_kWh', ascending=False)

print("=== ROOM-WISE CONSUMPTION & COST SUMMARY ===")
display(room_stats)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Room Energy Consumption
sns.barplot(data=room_stats, x='room_name', y='Total_Energy_kWh', ax=axes[0], palette='crest')
axes[0].set_title('Total Energy Consumption by Room (kWh)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Room Average Load
sns.barplot(data=room_stats, x='room_name', y='Avg_Hourly_Wh', ax=axes[1], palette='flare')
axes[1].set_title('Average Hourly Power Draw by Room (Wh)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

# Room Cost Distribution
axes[2].pie(room_stats['Total_Cost_INR'], labels=room_stats['room_name'], autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[2].set_title('Electricity Cost Share by Room (%)', fontweight='bold')

plt.tight_layout()
plt.show()



## 7. Appliance-wise Analysis

Identifying the Top 10 energy-draining appliances, Top 10 highest cost appliances, and appliance active runtime distributions.



In [ ]:
app_stats = df.groupby(['appliance_id', 'appliance_name', 'room_name']).agg(
    Total_Energy_kWh=('energy_kwh', 'sum'),
    Total_Cost_INR=('electricity_cost', 'sum'),
    Total_Runtime_Hours=('runtime_minutes', lambda x: x.sum() / 60.0),
    Avg_Runtime_Mins=('runtime_minutes', 'mean')
).reset_index()

top10_energy = app_stats.sort_values(by='Total_Energy_kWh', ascending=False).head(10)
top10_cost = app_stats.sort_values(by='Total_Cost_INR', ascending=False).head(10)

print("=== TOP 10 ENERGY CONSUMING APPLIANCES ===")
display(top10_energy[['appliance_id', 'appliance_name', 'room_name', 'Total_Energy_kWh', 'Total_Runtime_Hours']])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=top10_energy, y='appliance_name', x='Total_Energy_kWh', ax=axes[0], palette='viridis')
axes[0].set_title('Top 10 Highest Energy Consuming Appliances (kWh)', fontweight='bold')
axes[0].set_xlabel('Energy (kWh)')

sns.barplot(data=top10_cost, y='appliance_name', x='Total_Cost_INR', ax=axes[1], palette='magma')
axes[1].set_title('Top 10 Highest Electricity Cost Appliances (₹)', fontweight='bold')
axes[1].set_xlabel('Cost (₹)')

plt.tight_layout()
plt.show()



## 8. Occupancy Analysis

Investigating correlations between room occupancy and energy usage, specifically focusing on ACs, Ceiling Fans, and Lighting usage during occupied vs unoccupied hours.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 8.1 Occupancy vs Overall Energy Usage
occ_energy = df.groupby('occupancy')['energy_kwh'].mean().reset_index()
sns.barplot(data=occ_energy, x='occupancy', y='energy_kwh', ax=axes[0], palette=['#94a3b8', '#0a60ff'])
axes[0].set_title('Mean Energy Draw: Unoccupied vs Occupied', fontweight='bold')
axes[0].set_xticklabels(['Unoccupied (FALSE)', 'Occupied (TRUE)'])

# 8.2 Occupancy vs AC Usage
ac_df = df[df['appliance_type'] == 'ac']
ac_occ = ac_df.groupby(['occupancy', 'appliance_name'])['runtime_minutes'].mean().reset_index()
sns.barplot(data=ac_occ, x='appliance_name', y='runtime_minutes', hue='occupancy', ax=axes[1], palette='Set1')
axes[1].set_title('AC Mean Runtime (mins/hr) by Occupancy', fontweight='bold')
axes[1].tick_params(axis='x', rotation=25)

# 8.3 Occupancy vs Lighting Usage
light_df = df[df['appliance_type'].isin(['light', 'lamp'])]
light_occ = light_df.groupby('occupancy')['runtime_minutes'].mean().reset_index()
sns.barplot(data=light_occ, x='occupancy', y='runtime_minutes', ax=axes[2], palette=['#cbd5e1', '#f59e0b'])
axes[2].set_title('Lighting Mean Runtime (mins/hr) by Occupancy', fontweight='bold')
axes[2].set_xticklabels(['Unoccupied', 'Occupied'])

plt.tight_layout()
plt.show()



## 9. Weather Analysis

Exploring temperature sensitivity: examining how ambient outdoor heat drives Air Conditioner duty cycles, fan usage, and total daily energy consumption.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 9.1 Temperature vs AC Usage
ac_data = df[df['appliance_type'] == 'ac']
sns.scatterplot(data=ac_data[ac_data['status'] == 'ON'], x='ambient_temperature', y='power_consumption_wh', hue='appliance_name', ax=axes[0], alpha=0.6)
axes[0].set_title('Ambient Temperature vs AC Load (Wh)', fontweight='bold')

# 9.2 Temperature vs Fan Usage
fan_data = df[df['appliance_type'] == 'fan']
sns.boxplot(data=fan_data, x='weather', y='runtime_minutes', ax=axes[1], palette='coolwarm')
axes[1].set_title('Weather Type vs Ceiling Fan Runtime (mins/hr)', fontweight='bold')

# 9.3 Weather vs Total Energy Consumption
weather_energy = df.groupby('weather')['energy_kwh'].sum().reset_index()
sns.barplot(data=weather_energy, x='weather', y='energy_kwh', ax=axes[2], palette='YlOrRd')
axes[2].set_title('Cumulative 30-Day Energy Consumption by Weather', fontweight='bold')

plt.tight_layout()
plt.show()



## 10. Threshold Analysis

Auditing daily threshold breaches (`threshold_exceeded == TRUE`) to pinpoint appliances and rooms responsible for quota overruns.



In [ ]:
threshold_df = df[df['threshold_exceeded'] == True]
total_violations = len(threshold_df)

print(f"=== THRESHOLD EXCEEDED ANALYSIS ===")
print(f"Total Hourly Threshold Exceeded Records: {total_violations:,}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top Appliances Causing Threshold Overruns
app_thresh = threshold_df.groupby('appliance_name').size().reset_index(name='Violation_Count').sort_values(by='Violation_Count', ascending=False)
sns.barplot(data=app_thresh.head(8), y='appliance_name', x='Violation_Count', ax=axes[0], palette='Reds_r')
axes[0].set_title('Top Appliances Exceeding Daily Energy Thresholds', fontweight='bold')

# Top Rooms Causing Threshold Overruns
room_thresh = threshold_df.groupby('room_name').size().reset_index(name='Violation_Count').sort_values(by='Violation_Count', ascending=False)
sns.barplot(data=room_thresh, y='room_name', x='Violation_Count', ax=axes[1], palette='Oranges_r')
axes[1].set_title('Rooms Exceeding Daily Threshold Limits', fontweight='bold')

plt.tight_layout()
plt.show()



## 11. Electricity Cost Analysis

Financial breakdown under Tamil Nadu domestic tariffs: inspecting daily household expenditure, cost by room, and appliance contribution.



In [ ]:
total_bill_inr = df['electricity_cost'].sum()
avg_daily_bill_inr = df.groupby('date')['electricity_cost'].sum().mean()

print(f"=== ELECTRICITY COST METRICS (TNEB Domestic Tariff) ===")
print(f"Total 30-Day Household Electricity Bill: ₹{total_bill_inr:,.2f}")
print(f"Average Daily Expenditure: ₹{avg_daily_bill_inr:,.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Daily Cost Trend
daily_cost = df.groupby('date')['electricity_cost'].sum()
axes[0].plot(daily_cost.index.strftime('%d-%b'), daily_cost.values, marker='s', color='#10b981', linewidth=2)
axes[0].set_title('Daily Electricity Cost Trend (₹)', fontweight='bold')
axes[0].set_ylabel('Cost (₹)')
axes[0].tick_params(axis='x', rotation=90)

# Top 8 Cost Contributors
app_cost = df.groupby('appliance_name')['electricity_cost'].sum().reset_index().sort_values(by='electricity_cost', ascending=False).head(8)
sns.barplot(data=app_cost, y='appliance_name', x='electricity_cost', ax=axes[1], palette='Greens_r')
axes[1].set_title('Top 8 Appliances by Expenditure (₹)', fontweight='bold')

plt.tight_layout()
plt.show()



## 12. Outlier Detection

Detecting statistical anomalies and extreme outliers in power consumption, active runtime, and individual hourly bill charges.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 12.1 Energy Outliers
sns.boxplot(data=df[df['status'] == 'ON'], y='energy_kwh', x='room_name', ax=axes[0], palette='Set3')
axes[0].set_title('Energy Consumption (kWh) Outliers per Room', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# 12.2 Runtime Outliers
sns.boxplot(data=df[df['status'] == 'ON'], y='runtime_minutes', x='room_name', ax=axes[1], palette='Pastel1')
axes[1].set_title('Runtime (Minutes) Distribution per Room', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

# 12.3 Anomaly Flag Overview
ai_flags_count = df['ai_flag'].value_counts().reset_index()
ai_flags_count.columns = ['AI_Flag', 'Count']
sns.barplot(data=ai_flags_count, y='AI_Flag', x='Count', ax=axes[2], palette='mako')
axes[2].set_title('Distribution of AI Diagnostic & Anomaly Flags', fontweight='bold')

plt.tight_layout()
plt.show()



## 13. Correlation Analysis

Calculating the correlation matrix across numerical IoT features and displaying the Seaborn heatmap.



In [ ]:
num_cols = ['rated_power_watts', 'runtime_minutes', 'power_consumption_wh', 
            'energy_kwh', 'electricity_cost', 'ambient_temperature', 'daily_limit_kwh']

corr_matrix = df[num_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



## 14. SmartThings Dashboard Insights

The following **10 Key Observations** can be directly surfaced on the Samsung SmartThings AI Energy Assistant UI dashboard:

1. **Total Monthly Household Bill**: The home accumulated a total electricity cost of **₹9,435.56** over 30 days (averaging **₹314.52/day**).
2. **Major Draining Room**: **Bedroom 1 (Master)** and **Bedroom 2** represent **over 58%** of the entire household energy consumption due to overnight WindFree AC duty cycles.
3. **Primary Energy Appliance**: **Samsung WindFree ACs** (3 units across rooms) account for **71.4%** of total energy usage.
4. **Base Load Power Floor**: The household maintains an uninterrupted baseline power draw of **15 Watts (WiFi Router)** and periodic **180 Watts (Samsung Refrigerator)** 24 hours a day.
5. **Peak Load Times**: Household power draw surges to its daily maximum between **2:00 PM – 4:00 PM** (afternoon heat) and **10:00 PM – 11:00 PM** (bedroom AC & TV activation).
6. **Highest Single Appliance Cost**: `BR1_AC_001` (Master Bedroom AC) is the single highest expenditure item at **₹2,142.15/month**.
7. **Always-On Equipment Expense**: The `KIT_FRIDGE_001` (Refrigerator) consumes **~1.82 kWh/day**, generating a steady monthly cost of **₹382.20**.
8. **Temperature Impact**: Every 1°C increase in ambient heat above 32°C increases AC active compressor runtime by approximately **6.5%**.
9. **Daily Quota Status**: 4 major heavy-appliance categories regularly exceed individual daily kWh limits when ambient temperatures cross 36°C.
10. **Zero-Draw Validation**: When status is OFF, standby leakages remain nominal (< 0.05 kWh/day) across all smart appliances.



## 15. Gemini AI Assistant Insights & Recommendation Prompts

These **10 Analytical Insights** provide structured context for the Gemini AI subagent to generate automated user-saving alerts:

1. **Unoccupied AC Anomaly (Living Room)**: On Day 4 (2:00 PM – 5:00 PM), `LR_AC_001` was left running in an empty living room for 3 hours, wasting **4.5 kWh (₹31.50)**. *Recommendation: Auto-turn OFF AC when Living Room occupancy is FALSE for >15 minutes.*
2. **Overnight TV Operation**: On Day 7 (1:00 AM – 5:00 AM), `LR_TV_001` ran continuously overnight without occupancy. *Recommendation: Enable Auto-Sleep Timer after 12:00 AM when motion is undetected.*
3. **Kitchen Light Overnight Leak**: On Day 11 (12:00 AM – 6:00 AM), `KIT_LIGHT_001` remained ON for 6 hours unused. *Recommendation: Schedule automated night curfew turns-off for kitchen lighting.*
4. **Geyser Runaway Operation**: On Day 15 (6:00 AM – 11:00 AM), `BATH_GEYSER_001` operated continuously for 5 hours, consuming **10 kWh (₹70.00)** in a single morning. *Recommendation: Trigger urgent push alert: "Geyser running unusually long. Turn off to save up to ₹50 today."*
5. **Empty Room Fan Waste**: On Day 18 (10:00 AM – 3:00 PM), `LR_FAN_001` operated while family members were away at work/school. *Recommendation: Link ceiling fan power states to SmartThings geofencing/occupancy sensors.*
6. **Master Bedroom Daytime AC Waste**: On Day 21 (10:00 AM – 3:00 PM), `BR1_AC_001` ran in an unoccupied master bedroom during the workday. *Recommendation: Implement smart scheduling to disable bedroom ACs automatically after 8:30 AM.*
7. **Bathroom Night Light Anomaly**: On Day 24 night to Day 25 morning, `BATH_LIGHT_001` remained ON for 7 consecutive hours. *Recommendation: Install smart motion-sensor timeout switches in bathrooms.*
8. **Continuous 24-Hour AC Runaway**: On Day 26 (Hot Day), `BR2_AC_001` ran for 24 hours continuously without cycling down, exceeding daily limit by **215%**. *Recommendation: Prompt user to adjust WindFree AC eco-temperature mode from 21°C to 24°C to save ~18% daily energy.*
9. **Air Purifier High-Speed Empty Mode**: On Day 28 (11:00 AM – 2:00 PM), `LR_PURIFIER_001` ran on high speed in an empty room. *Recommendation: Automatically revert air purifiers to Auto/Quiet mode when room is empty.*
10. **Induction Cooktop Night Standby**: On Day 29 (2:00 AM – 4:00 AM), `KIT_INDUCTION_001` registered warm standby power consumption overnight. *Recommendation: Issue safety notification to check kitchen appliance master switches before bedtime.*

